In [1]:
# DATA EXTRACTION AND CLEANING
#
# Starts from df_selected, the dataset produced by 1b-DataPreparation: the applications
# retrieved by keywords, IPC or CPC, in families with at least one patent of invention
# and a priority year within the window (both filters are applied in 1b).
#
# In this notebook we will:
# 1. Retrieve all available members of the selected DOCDB families.
# 2. Build a family-level table with the earliest priority year.
# 3. Keep only the patents of invention within each family and compute family aggregates.
# 4. Extract applicants and countries, preserving their links to applications.
# 5. Retrieve classifications, texts, citations and legal events for the later steps.
#
# Family expansion does not reapply the keyword, classification or filing-year filters.

from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sector.yaml").exists())
cfg = yaml.safe_load((ROOT / "config" / "sector.yaml").read_text(encoding="utf-8"))

input_file = Path(cfg["paths"]["df_selected"])     # written by 1b-DataPreparation

if not input_file.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {input_file.resolve()}. "
        "Export df_selected from DataPreparation first, "
        "or update input_file to its location."
    )

df_selected = pd.read_parquet(input_file)

required_columns = {"appln_id", "docdb_family_id", "retrieval_source"}
missing_columns = required_columns - set(df_selected.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if df_selected["appln_id"].isna().any():
    raise ValueError("The input contains missing application IDs.")

if df_selected["appln_id"].duplicated().any():
    raise ValueError("The input contains duplicate application IDs.")

print("PROVISIONAL INPUT DATASET")
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {df_selected['docdb_family_id'].nunique():,}")
print(f"Columns: {len(df_selected.columns)}")
print("Validation status: pending")

print("\nDATA PREVIEW")
display(df_selected.head())

# Show column names, data types, missing values and blank text values.
schema = pd.DataFrame({
    "column": df_selected.columns,
    "dtype": [str(df_selected[c].dtype) for c in df_selected.columns],
    "missing_values": [
        int(df_selected[c].isna().sum())
        for c in df_selected.columns
    ],
    "blank_strings": [
        int(df_selected[c].map(
            lambda value: isinstance(value, str) and not value.strip()
        ).sum())
        for c in df_selected.columns
    ],
})

print("\nCOLUMN OVERVIEW")
display(schema)

print("\nAPPLICATIONS BY RETRIEVAL SOURCE")
display(
    df_selected["retrieval_source"]
    .value_counts(dropna=False)
    .rename_axis("retrieval_source")
    .reset_index(name="applications")
)

# Keep the imported selection intact.
# Expanded family members and cleaned tables will use separate DataFrames.

PROVISIONAL INPUT DATASET
Applications: 150,487
DOCDB families: 115,491
Columns: 12
Validation status: pending

DATA PREVIEW


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract,sbert_similarity,retrieval_source
0,2442,38192468,EP,06838485,A,2006-11-29,2005,Y,ELECTRIC MOTOR ASSISTED TAKEOFF DEVICE FOR AN ...,"A takeoff assist device for an air vehicle, su...",0.733866,initial_union
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...,0.892258,initial_union
2,28800,39609207,WO,2007069569,W,2007-05-23,2006,N,DUAL-USE MODULAR PROPULSION SURVEILLANCE VEHIC...,The present invention provides a system for re...,0.944573,initial_union
3,28822,39609212,WO,2007073846,W,2007-07-19,2006,N,AERODYNAMIC INTEGRATION OF A PAYLOAD CONTAINER...,A vertical takeoff and landing (VTOL) rotary-w...,0.833960,initial_union
4,36005,37810003,WO,2008000133,W,2008-01-16,2007,N,APPARATUS FOR A HIVE,A trap(1) for attachment to a hive entrance co...,0.331044,initial_union



COLUMN OVERVIEW


,column,dtype,missing_values,blank_strings
0,appln_id,int64,0,0
1,docdb_family_id,int64,0,0
2,appln_auth,object,0,0
3,appln_nr,object,0,0
4,appln_kind,object,0,0
5,appln_filing_date,object,0,0
6,earliest_filing_year,int64,0,0
7,granted,object,0,0
8,appln_title,object,0,36
9,appln_abstract,object,0,0



APPLICATIONS BY RETRIEVAL SOURCE


,retrieval_source,applications
0,initial_union,150487


## 1. Setup for extraction
Connects to PATSTAT and prepares the list of selected families

In [2]:
from sqlalchemy import func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
    TLS206_PERSON, TLS207_PERS_APPLN,
    TLS211_PAT_PUBLN, TLS212_CITATION, TLS215_CITN_CATEG,
    TLS224_APPLN_CPC, TLS231_INPADOC_LEGAL_EVENT
)

# configuration (config/sector.yaml)
ENV = cfg["env"]                                   # must match the environment df_selected was built in
OUTPUT_DIR = Path(cfg["paths"]["extract_dir"]) / ENV.lower()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = cfg["years"]["min"], cfg["years"]["max"]   # family priority years (filtered in 1b, checked here)
ipr_cfg = cfg["counting"]["ipr_type"]
KEEP_IPR_TYPES = (ipr_cfg,) if isinstance(ipr_cfg, str) else tuple(ipr_cfg)   # e.g. ("PI",)
CITATIONS_EP_ONLY = True                           # tutorial_3/30: EP citation practice is consistent
CHUNK = 5000                                       # ids per IN (...) list

patstat = PatstatClient(env=ENV)
db = patstat.orm()


def strip_text_columns(df):
    """Remove padding spaces from all text columns (PATSTAT pads codes like 'A ', 'EP ')."""
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip() if df[col].map(lambda v: isinstance(v, str)).any() else df[col]
    return df


def query_in_chunks(build_query, ids):
    """Run build_query(list_of_ids) on chunks of ids, stack the results, and clean text."""
    ids = [int(i) for i in pd.unique(pd.Series(ids).dropna())]
    parts = [patstat.df(build_query(ids[s:s + CHUNK])) for s in range(0, len(ids), CHUNK)]
    parts = [p for p in parts if len(p)]
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return strip_text_columns(df)


def save(df, name):
    df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    print(f"{name:<22} {len(df):>10,} rows  ->  {OUTPUT_DIR / name}.parquet")


# families of the selection (unit of analysis: DOCDB family)
fid = pd.to_numeric(df_selected["docdb_family_id"], errors="coerce")
family_ids = sorted({int(f) for f in fid[fid.gt(0)]})
print(f"Valid family ids: {len(family_ids):,}")

Valid family ids: 115,491


## 2. TLS201: all applications of each family

**Step 1: Collect all applications of each invention.**
One invention (a *family*) is often filed in several countries, for example in China, the USA and Europe.
Our search found only some of these filings. Here we collect **all** the filings of each invention,
because useful information (grant, citations, English abstract) can be on any of them.

**Step 2: Give each invention one year.**
Each filing records the year the invention was **first** filed anywhere.
We take the **earliest** of these years as the invention's year (the *priority year*).
PATSTAT writes `9999` when the year is unknown; we ignore those.

**Step 3: Check that all inventions are within the chosen years** (`YEAR_MIN` to `YEAR_MAX`; the filter is applied in 1b-DataPreparation).

In [3]:
appln = query_in_chunks(lambda ch: db.query(
    TLS201_APPLN.appln_id, TLS201_APPLN.docdb_family_id, TLS201_APPLN.inpadoc_family_id,
    TLS201_APPLN.appln_auth, TLS201_APPLN.appln_nr, TLS201_APPLN.appln_kind, TLS201_APPLN.ipr_type,
    TLS201_APPLN.appln_filing_date, TLS201_APPLN.appln_filing_year,
    TLS201_APPLN.earliest_filing_year, TLS201_APPLN.earliest_publn_year,
    TLS201_APPLN.granted, TLS201_APPLN.docdb_family_size, TLS201_APPLN.nb_citing_docdb_fam,
    TLS201_APPLN.nb_applicants, TLS201_APPLN.nb_inventors,
).filter(TLS201_APPLN.docdb_family_id.in_(ch)), family_ids)

year = pd.to_numeric(appln["earliest_filing_year"], errors="coerce")
appln["valid_year"] = year.where(year.between(1800, 9998))     # 9999 = unknown year

families = (appln.groupby("docdb_family_id")
            .agg(priority_year=("valid_year", "min"))
            .reset_index())
appln = appln.drop(columns="valid_year")

# The priority-year filter is applied in 1b (section 1.1): here it is only checked.
outside = ~families["priority_year"].between(YEAR_MIN, YEAR_MAX)
assert not outside.any(), (f"{int(outside.sum())} families outside {YEAR_MIN}-{YEAR_MAX}: "
                           "df_selected does not come from the current 1b. Rerun 1b first.")
families["priority_year"] = families["priority_year"].astype(int)

families["priority_year"].value_counts().sort_index()

priority_year
2000       73
2001       87
2002       86
2003      131
2004      110
2005      143
2006      180
2007      183
2008      252
2009      305
2010      357
2011      448
2012      580
2013      870
2014     1720
2015     3641
2016     7114
2017     9733
2018    10538
2019    10301
2020    11505
2021    12285
2022    12874
2023    14958
2024    17017
Name: count, dtype: int64

**Patents of invention only.** Families are filtered in 1b (section 1.1): a family stays if it has at least one patent of invention (`PI`). Here the utility-model (`UM`) and design (`DP`) members of the kept families are removed, and the family aggregates (applications, offices, grant) are computed on patents of invention only. The families excluded in 1b are listed in `df_family_filter.parquet`.

In [4]:
appln.groupby([appln["appln_auth"].str.strip(), appln["ipr_type"].str.strip()]).size().unstack(fill_value=0).sort_values("PI", ascending=False).head(10)

ipr_type,DP,PI,UM
appln_auth,,,
CN,0,88487,702
US,13,22586,0
WO,0,13483,0
KR,0,11639,5
JP,0,7022,21
EP,0,6772,0
AU,0,1823,0
CA,0,1524,0
RU,0,1063,0


In [5]:
# Patents of invention only (counting.ipr_type in config/sector.yaml).
# Families are already filtered in 1b (section 1.1): a family stays if it has at least one
# patent of invention. Here the other members of those families (utility models, designs)
# are removed, so that the indicators (offices, grants, citations) use patents of invention only.
# - DP (designs): protect appearance, not technology (tutorial 10)
# - UM (utility models): mostly Chinese, usually not examined; would inflate
#   China's share (tutorial 10 §3.2-3.3); tutorial_1/11 also uses IPR_TYPE = "PI"
ipr = appln["ipr_type"].astype(str).str.strip()
print("Applications by IPR type, before filtering:")
display(ipr.value_counts())

excluded = appln[~ipr.isin(KEEP_IPR_TYPES)].copy()
save(excluded, "excluded_ipr_types")           # members removed from the kept families
print("Excluded applications by office:")
display(excluded.groupby([excluded["appln_auth"].str.strip(), excluded["ipr_type"].str.strip()]).size())

appln = appln[ipr.isin(KEEP_IPR_TYPES)].copy()
appln_ids = appln["appln_id"].tolist()

# Consistency check: every family must still have a patent of invention.
no_patent = set(families["docdb_family_id"]) - set(appln["docdb_family_id"])
assert not no_patent, (f"{len(no_patent)} families without a {'/'.join(KEEP_IPR_TYPES)} member: "
                       "df_selected does not come from the current 1b. Rerun 1b first.")

# Family-level aggregates, computed on the patents of invention only.
aggregates = (appln.groupby("docdb_family_id")
              .agg(n_applications=("appln_id", "nunique"),
                   n_offices=("appln_auth", "nunique"),
                   granted_any=("granted", lambda g: (g.astype(str).str.strip() == "Y").any()))
              .reset_index())
families = families.merge(aggregates, on="docdb_family_id", how="left", validate="one_to_one")

print(f"Applications removed (non-{'/'.join(KEEP_IPR_TYPES)} members): {len(excluded):,}  (kept {len(appln):,})")
save(families, "families")
save(appln, "tls201_appln")

Applications by IPR type, before filtering:


ipr_type
PI    162141
UM       872
DP        13
Name: count, dtype: int64

excluded_ipr_types            885 rows  ->  /home/jovyan/progetto/extract/prod/excluded_ipr_types.parquet
Excluded applications by office:


appln_auth  ipr_type
AT          UM            6
BR          UM            1
CN          UM          702
CZ          UM            1
DE          UM           75
DK          UM            1
ES          UM           20
FR          UM            3
HR          UM            1
HU          UM            1
IE          UM            7
IT          UM            5
JP          UM           21
KR          UM            5
PE          UM            1
PL          UM            1
SK          UM            1
TW          UM           13
UA          UM            7
US          DP           13
dtype: int64

Applications removed (non-PI members): 885  (kept 162,141)
families                  115,491 rows  ->  /home/jovyan/progetto/extract/prod/families.parquet
tls201_appln              162,141 rows  ->  /home/jovyan/progetto/extract/prod/tls201_appln.parquet


## 3. `TLS224_APPLN_CPC`: all technology classes

Each patent has **many** technology codes, not just one.
For example, a delivery drone can have a code for "drone", one for "flight control" and one for "sensors".

In Step 0 we used only a few codes (B64U, B64C39/024) to **find** the drone patents. Here we take **all** the codes of these patents, because the codes are used in almost every later step.

We also remove the spaces inside the codes (`B64U  10/13` → `B64U10/13`) so they are easier to compare.

In [6]:
cpc = query_in_chunks(lambda ch: db.query(
    TLS224_APPLN_CPC.appln_id,
    func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("cpc"),
).filter(TLS224_APPLN_CPC.appln_id.in_(ch)).distinct(), appln_ids)

cpc = cpc.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
cpc["subclass"] = cpc["cpc"].str[:4]                 # e.g. B64U
cpc["main_group"] = cpc["cpc"].str.split("/").str[0] # e.g. B64U10
save(cpc, "tls224_cpc")
cpc.drop_duplicates(["docdb_family_id", "subclass"])["subclass"].value_counts().head(10)

tls224_cpc              1,212,309 rows  ->  /home/jovyan/progetto/extract/prod/tls224_cpc.parquet


subclass
B64U    38811
B64C    25502
B64D    21914
G05D    19219
Y02T    13555
G06V     8786
G06T     7835
G06F     7335
G01C     6924
G01S     6768
Name: count, dtype: int64

In [7]:
# Official CPC subclass titles – source: cooperativepatentclassification.org (CPC scheme)
CPC_SUBCLASS_TITLES = {
    "B64C": "Aeroplanes; helicopters",
    "B64U": "Unmanned aerial vehicles [UAV]; equipment therefor",
    "G05D": "Systems for controlling or regulating non-electric variables",
    "B64D": "Equipment for fitting in or to aircraft; flight suits; parachutes; "
            "arrangement or mounting of power plants or propulsion transmissions in aircraft",
    "Y02E": "Reduction of greenhouse gas [GHG] emissions, related to energy generation, "
            "transmission or distribution",
    "Y02T": "Climate change mitigation technologies related to transportation",
    "B64F": "Ground installations or apparatus specially adapted for use on aircraft; "
            "designing, manufacturing, assembling, maintaining or repairing aircraft",
    "F03D": "Wind motors",
    "G08G": "Traffic control systems",
    "G01S": "Radio direction-finding; radio navigation; determining distance or velocity "
            "by use of radio waves; locating or presence-detecting by reflection of radio waves",
}

n_fam = families["docdb_family_id"].nunique()
top = (cpc.drop_duplicates(["docdb_family_id", "subclass"])["subclass"]
         .value_counts().head(10).rename("families").to_frame())
top["share"] = (top["families"] / n_fam * 100).round(1).astype(str) + "%"
top["definition"] = top.index.map(CPC_SUBCLASS_TITLES).fillna("⚠️ add title")
pd.set_option("display.max_colwidth", None)   # show full text in every cell
top

,families,share,definition
subclass,,,
B64U,38811,33.6%,Unmanned aerial vehicles [UAV]; equipment therefor
B64C,25502,22.1%,Aeroplanes; helicopters
B64D,21914,19.0%,Equipment for fitting in or to aircraft; flight suits; parachutes; arrangement or mounting of power plants or propulsion transmissions in aircraft
G05D,19219,16.6%,Systems for controlling or regulating non-electric variables
Y02T,13555,11.7%,Climate change mitigation technologies related to transportation
G06V,8786,7.6%,⚠️ add title
G06T,7835,6.8%,⚠️ add title
G06F,7335,6.4%,⚠️ add title
G01C,6924,6.0%,⚠️ add title


## 4. `TLS206_PERSON` + `TLS207_PERS_APPLN`: applicants and inventors

Each patent lists **people and companies**. They can have two roles:
- **Applicant** = the **owner** of the patent (usually a company or university).
- **Inventor** = the **person** who made the invention.
One person can be both (for example, a small inventor who files alone).

PATSTAT writes the same company in many ways (for example "SIEMENS AG" and "Siemens Aktiengesellschaft").
`psn_name` is PATSTAT's cleaned version of the name.

*-- waiting to implement name harmonization --*

We also keep the **address and region** of each person, to make maps of where inventors work (tutorial 3).

**Notes** : From PATSTAT Global manual 2026, “PSN attributes are not updated since 2021 Autumn edition.” PSN = PATSTAT Standardised Name, i.e. `psn_name`, `psn_id` and `psn_sector`.

In [8]:
persons = query_in_chunks(lambda ch: db.query(
    TLS207_PERS_APPLN.appln_id, TLS207_PERS_APPLN.person_id,
    TLS207_PERS_APPLN.applt_seq_nr, TLS207_PERS_APPLN.invt_seq_nr,
    TLS206_PERSON.person_name, TLS206_PERSON.psn_id, TLS206_PERSON.psn_name, TLS206_PERSON.psn_sector,
    TLS206_PERSON.han_id, TLS206_PERSON.han_name,
    func.trim(TLS206_PERSON.person_ctry_code).label("ctry"),
    TLS206_PERSON.person_address, TLS206_PERSON.nuts, TLS206_PERSON.nuts_level,
).join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
 .filter(TLS207_PERS_APPLN.appln_id.in_(ch)).distinct(), appln_ids)

persons = persons.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
persons["is_applicant"] = persons["applt_seq_nr"] > 0
persons["is_inventor"] = persons["invt_seq_nr"] > 0
save(persons, "tls206_207_persons")

applicants = persons[persons["is_applicant"]]
print("Families without any applicant:",
      len(set(families["docdb_family_id"]) - set(applicants["docdb_family_id"])))
(applicants.drop_duplicates(["docdb_family_id", "psn_name"])["psn_name"]
 .value_counts().head(15).rename("families"))

tls206_207_persons        766,421 rows  ->  /home/jovyan/progetto/extract/prod/tls206_207_persons.parquet
Families without any applicant: 138


psn_name
SZ DJI TECHNOLOGY COMPANY                             2110
SGCC(STATE GRID CORPORATION OF CHINA)                 1312
NANJING UNIVERSITY OF AERONAUTICS AND ASTRONAUTICS    1276
BEIHANG UNIVERSITY                                    1269
NORTHWESTERN POLYTECHNICAL UNIVERSITY                 1079
CHINESE ACADEMY OF SCIENCES                            787
GUANGDONG POWER GRID CORPORATION                       763
BEIJING INSTITUTE OF TECHNOLOGY                        729
AUTEL ROBOTICS COMPANY                                 530
HARBIN INSTITUTE OF TECHNOLOGY                         495
LG ELECTRONICS                                         460
ZHEJIANG UNIVERSITY                                    429
BOEING COMPANY                                         426
NATIONAL UNIVERSITY OF DEFENSE TECHNOLOGY OF PLA       412
XIDIAN UNIVERSITY                                      399
Name: families, dtype: int64

In [9]:
app = persons[persons["is_applicant"]]
print("applicant rows:", len(app))
print("with psn_name:", app["psn_name"].fillna("").str.strip().ne("").sum())
print("with han_name:", app["han_name"].fillna("").str.strip().ne("").sum())

# same company, different names? first 10 examples
app[["person_name", "psn_name", "han_name"]].drop_duplicates().head(10)

applicant rows: 184319
with psn_name: 184319
with han_name: 184305


,person_name,psn_name,han_name
0,"INSTITUTE OF AUTOMATION, CHINESE ACADEMY OF SCIENCES",CHINESE ACADEMY OF SCIENCES,SHANGHAI INSTITUTE OF MATERIA MEDICA CHINESE ACADEMY OF SCIENCES
1,INSTITUTE OF MICROELECTRONICS OF CHINESE ACADEMY OF SCIENCES,CHINESE ACADEMY OF SCIENCES,SHANGHAI INSTITUTE OF MATERIA MEDICA CHINESE ACADEMY OF SCIENCES
2,Board of Trustees of Michigan State University,MICHIGAN STATE UNIVERSITY,BOARD OF TRUSTEES OF MICHIGAN STATE UNIVERSITY
3,The United States of America as represented by the Administrator of the National Aeronautics and Space Administration,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTRATION),THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
5,The United States of America as represented by the Administration of NASA,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTRATION),THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
6,"UNITED STATES OF AMERICA, AS REPRESENTED BY THE ADMINISTRATOR OF THE NATIONAL AERONAUTICS AND SPACE ADMINISTRATION",NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTRATION),THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
7,The United States of America as represented by the Administrator of the,THE UNITED STATES OF AMERICA AS REPRESENTED BY THE ADMINISTRATOR,THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
8,The United States of America as represented by the Administrator of the National Aeronautics and Spa,NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTRATION),THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
9,"The United States of America, as represented by the Administrator of the National Aeronautics and Space Administration",NASA (NATIONAL AERONAUTICS AND SPACE ADMINISTRATION),THE UNITED STATES OF AMERICA AS REPRESENTED BY THE SECRETARY DEPARTMENT OF HEALTH & HUMAN SERVICES
10,21ST CENTURY AIRSHIPS INC.,21ST CENTURY AIRSHIPS,21ST CENTURY AIRSHIPS INC


In [10]:
no_app = set(families["docdb_family_id"]) - set(persons.loc[persons["is_applicant"], "docdb_family_id"])
appln[appln["docdb_family_id"].isin(no_app)]["appln_auth"].str.strip().value_counts()

appln_auth
KR    85
US    34
RU    11
TW    10
Name: count, dtype: int64

In [11]:
fam = appln[appln["docdb_family_id"].isin(no_app)]
print(fam["appln_filing_year"].value_counts().sort_index())   # mostly before ~2013?
print(fam["docdb_family_size"].value_counts())                 # single-office families?

# do they at least have inventors?
inv = persons[persons["docdb_family_id"].isin(no_app) & persons["is_inventor"]]
print("families with inventors:", inv["docdb_family_id"].nunique(), "of", len(no_app))

appln_filing_year
2001     1
2002     5
2003     3
2004     9
2005    14
2006     6
2007     8
2008     1
2009     1
2010     1
2011     1
2012     2
2013     5
2014     1
2015     2
2016     2
2018     1
2019     1
2020    48
2021    24
2022     2
2023     2
Name: count, dtype: int64
docdb_family_size
1    136
2      4
Name: count, dtype: int64
families with inventors: 53 of 138


## 5. `TLS202_APPLN_TITLE` / `TLS203_APPLN_ABSTR`: titles and abstracts

We get the **title** and the **abstract** (short summary) of every patent, and the **language** they are written in.

Later we analyse the text with language tools (for example to find topics or new ideas).
These tools need **one English text per invention**.
So for each invention (family) we pick **one English abstract**, from any of its patents.

In [12]:
text = query_in_chunks(lambda ch: db.query(
    TLS201_APPLN.appln_id,
    TLS202_APPLN_TITLE.appln_title, TLS202_APPLN_TITLE.appln_title_lg,
    TLS203_APPLN_ABSTR.appln_abstract, TLS203_APPLN_ABSTR.appln_abstract_lg,
).outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
 .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
 .filter(TLS201_APPLN.appln_id.in_(ch)), appln_ids)

text = text.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
save(text, "tls202_203_text")

en = text[(text["appln_abstract_lg"].str.strip() == "en") & text["appln_abstract"].notna()]
family_text = en.sort_values("appln_id").drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_title", "appln_abstract"]]
save(family_text, "family_abstract_en")
print(f"Families with an English abstract: {len(family_text):,} of {len(families):,}")

tls202_203_text           162,141 rows  ->  /home/jovyan/progetto/extract/prod/tls202_203_text.parquet
family_abstract_en        115,487 rows  ->  /home/jovyan/progetto/extract/prod/family_abstract_en.parquet
Families with an English abstract: 115,487 of 115,491


## 6. Citations: `TLS211` → `TLS212` (+ `TLS215`)

A **citation** is when a patent refers to an older patent ("my invention builds on this one").

**Why we need an extra table (TLS211).**
A patent application is **published several times**: first when it is filed (with a list of related older patents),
then again when it is granted, and so on. Each publication can add citations.
Citations are attached to these **publications**, not to the application itself.
So we go in three steps: application → its publications (TLS211) → their citations (TLS212),
and then we bring all citations back to the application.

**We use only European (EP) patents.**
We count citations only for EP patents (`CITATIONS_EP_ONLY = True`) - Tutorial 3

### 6.1 Publications (`TLS211`)

In [13]:
cit_appln = appln[appln["appln_auth"].str.strip() == "EP"] if CITATIONS_EP_ONLY else appln

pubs = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id, TLS211_PAT_PUBLN.appln_id, TLS211_PAT_PUBLN.publn_auth,
    TLS211_PAT_PUBLN.publn_nr, TLS211_PAT_PUBLN.publn_kind, TLS211_PAT_PUBLN.publn_date,
    TLS211_PAT_PUBLN.publn_first_grant,
).filter(TLS211_PAT_PUBLN.appln_id.in_(ch)).distinct(), cit_appln["appln_id"])

save(pubs, "tls211_publications")
print(f"{cit_appln['docdb_family_id'].nunique():,} families have an application in scope for citations")
pubs["publn_kind"].value_counts()

tls211_publications        13,199 rows  ->  /home/jovyan/progetto/extract/prod/tls211_publications.parquet
6,020 families have an application in scope for citations


publn_kind
A1    6017
B1    3400
A4    2171
A2     755
C0     438
A3     353
B8      50
B2       9
B9       3
A8       3
Name: count, dtype: int64

In [14]:
# EP publication kind codes – source: EPO kind codes (WIPO ST.16); verify on epo.org
PUBLN_KIND_LABELS = {
    "A1": "Application published WITH search report",
    "A2": "Application published WITHOUT search report",
    "A3": "Search report published later, separately",
    "A4": "Supplementary search report (PCT application entering Europe)",
    "A8": "Corrected title page of an A document",
    "A9": "Complete reprint of an A document",
    "B1": "Granted patent",
    "B2": "Granted patent, amended after opposition",
    "B3": "Granted patent, amended after limitation",
    "B8": "Corrected title page of a granted patent",
    "B9": "Complete reprint of a granted patent",
}

kind_table = pubs["publn_kind"].str.strip().value_counts().rename("publications").to_frame()
kind_table["share"] = (kind_table["publications"] / kind_table["publications"].sum() * 100).round(1)
kind_table["meaning"] = kind_table.index.map(PUBLN_KIND_LABELS).fillna("⚠️ check EPO kind codes")
kind_table.style.set_properties(subset=["meaning"], **{"text-align": "left"})

,publications,share,meaning
publn_kind,,,
A1,6017,45.600000,Application published WITH search report
B1,3400,25.800000,Granted patent
A4,2171,16.400000,Supplementary search report (PCT application entering Europe)
A2,755,5.700000,Application published WITHOUT search report
C0,438,3.300000,⚠️ check EPO kind codes
A3,353,2.700000,"Search report published later, separately"
B8,50,0.400000,Corrected title page of a granted patent
B2,9,0.100000,"Granted patent, amended after opposition"
B9,3,0.000000,Complete reprint of a granted patent


In [15]:
ep_fams = set(cit_appln["docdb_family_id"])
app = persons[persons["is_applicant"]].drop_duplicates(["docdb_family_id", "psn_name"])
app["has_ep"] = app["docdb_family_id"].isin(ep_fams)

top = (app.groupby("psn_name")
          .agg(families=("docdb_family_id", "nunique"), with_ep=("has_ep", "sum"))
          .sort_values("families", ascending=False).head(15))
top["ep_coverage"] = (top["with_ep"] / top["families"] * 100).round(0).astype(str) + "%"
top

,families,with_ep,ep_coverage
psn_name,,,
SZ DJI TECHNOLOGY COMPANY,2110,223,11.0%
SGCC(STATE GRID CORPORATION OF CHINA),1312,1,0.0%
NANJING UNIVERSITY OF AERONAUTICS AND ASTRONAUTICS,1276,0,0.0%
BEIHANG UNIVERSITY,1269,0,0.0%
NORTHWESTERN POLYTECHNICAL UNIVERSITY,1079,0,0.0%
CHINESE ACADEMY OF SCIENCES,787,1,0.0%
GUANGDONG POWER GRID CORPORATION,763,0,0.0%
BEIJING INSTITUTE OF TECHNOLOGY,729,0,0.0%
AUTEL ROBOTICS COMPANY,530,79,15.0%


### 6.2 Backward citations (`TLS212`) and their category (`TLS215`)

Here we collect the **older patents** that our drone patents refer to (like a bibliography).

For each reference we also keep **who added it** (`citn_origin`):
- `SEA` / `ISR` = the **patent office** found it when searching for similar older patents (most common in Europe)
- `APP` = the **applicant** (the company) added it
- `EXA` / `OPP` = added later, during **examination** or when someone **challenged** the patent

Some references point to documents PATSTAT **doesn't fully know** (no date, no family).
PATSTAT gives them special numbers starting from 900,000,001.
We **keep** them (so counts stay correct) but **mark** them, so we can leave them out where needed.

In [16]:
back = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.appln_id.label("citing_appln_id"),
    TLS212_CITATION.pat_publn_id, TLS212_CITATION.citn_replenished, TLS212_CITATION.citn_id,
    TLS212_CITATION.citn_origin, TLS212_CITATION.cited_pat_publn_id,
).join(TLS212_CITATION, TLS212_CITATION.pat_publn_id == TLS211_PAT_PUBLN.pat_publn_id)
 .filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch), TLS212_CITATION.cited_pat_publn_id > 0)
 .distinct(), pubs["pat_publn_id"])

cited = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id.label("cited_pat_publn_id"),
    TLS211_PAT_PUBLN.publn_auth.label("cited_auth"),
    TLS211_PAT_PUBLN.publn_date.label("cited_date"),
    TLS211_PAT_PUBLN.appln_id.label("cited_appln_id"),
).filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch)), back["cited_pat_publn_id"])

back = back.merge(cited, on="cited_pat_publn_id", how="left")
back["cited_artificial"] = back["cited_pat_publn_id"] >= 900_000_001
save(back, "tls212_backward")

# TLS215 has one row per category and per relevant claim: distinct keeps one row
# per (citation, category), otherwise categories are counted once per claim.
categ = query_in_chunks(lambda ch: db.query(
    TLS215_CITN_CATEG.pat_publn_id, TLS215_CITN_CATEG.citn_replenished,
    TLS215_CITN_CATEG.citn_id, TLS215_CITN_CATEG.citn_categ,
).filter(TLS215_CITN_CATEG.pat_publn_id.in_(ch)).distinct(), back["pat_publn_id"])
save(categ, "tls215_citn_categ")

back["citn_origin"].value_counts()

tls212_backward            48,432 rows  ->  /home/jovyan/progetto/extract/prod/tls212_backward.parquet
tls215_citn_categ          53,505 rows  ->  /home/jovyan/progetto/extract/prod/tls215_citn_categ.parquet


citn_origin
ISR    20665
SEA    17880
APP     7701
EXA     1780
FOP      252
TPO      111
OPP       43
Name: count, dtype: int64

In [17]:
# who added the citation
# Sources: PATSTAT Global manual 2026 (CITN_ORIGIN), tutorial 30
CITN_ORIGIN_LABELS = {   # source: PATSTAT Global manual 2026, CITN_ORIGIN
    "APP": "Added by the applicant",
    "SEA": "Examiner search (search report)",
    "ISR": "International Search Report (PCT)",
    "SUP": "Supplementary Search Report",
    "PRS": "US pre-search (USPTO examiner)",
    "EXA": "Added during examination",
    "OPP": "Opposition documents selected by the opposition division",
    "APL": "Added during an appeal",
    "FOP": "Opposition filed: added by the opponent or the proprietor",
    "TPO": "Third-party observation (Art. 115 EPC)",
    "CH2": "Added during PCT Chapter II",
}

origin_table = back["citn_origin"].str.strip().value_counts().rename("citations").to_frame()
origin_table["share"] = (origin_table["citations"] / origin_table["citations"].sum() * 100).round(1)
origin_table["meaning"] = origin_table.index.map(CITN_ORIGIN_LABELS).fillna("⚠️ check manual")
display(origin_table)

,citations,share,meaning
citn_origin,,,
ISR,20665,42.7,International Search Report (PCT)
SEA,17880,36.9,Examiner search (search report)
APP,7701,15.9,Added by the applicant
EXA,1780,3.7,Added during examination
FOP,252,0.5,Opposition filed: added by the opponent or the proprietor
TPO,111,0.2,Third-party observation (Art. 115 EPC)
OPP,43,0.1,Opposition documents selected by the opposition division


In [18]:
back_cat = back.merge(
    categ,
    on=["pat_publn_id", "citn_replenished", "citn_id"],
    how="left",
)
print(back_cat["citn_categ"].value_counts(dropna=False).head(10))   # X, Y, A …

citn_categ
A      20400
Y      11313
X      11148
NaN     9943
I       5984
AD       273
PX       236
E        205
XP       140
AP        76
Name: count, dtype: int64


In [19]:
# how relevant the cited document is
# Sources: PATSTAT Global manual 2026 (CITN_CATEG), search-report categories
CITN_CATEG_LABELS = {
    "X": "Very relevant on its own",
    "Y": "Relevant combined with another document",
    "A": "Background",
    "D": "Already cited in the application",
    "P": "Published between priority and filing date",
    "E": "Earlier patent published on/after filing date",
    "L": "Cited for other reasons",
    "O": "Non-written disclosure",
    "T": "Theory or principle behind the invention",
    "&": "Member of the same patent family",
    "I": "⚠️ check DOCDB Annex XIV",
}

def explain_categ(code):
    """Explain each letter of a (possibly multi-letter) category, e.g. 'XP'."""
    code = str(code).strip()
    return " + ".join(CITN_CATEG_LABELS.get(letter, f"⚠️ {letter}?") for letter in code)

# Categories of the BACKWARD PATENT citations only (back_cat = back + categ)
categ_table = (
    back_cat["citn_categ"]
    .str.strip()
    .fillna("—")                      # citations with no category (e.g. added by the applicant)
    .value_counts()
    .rename("rows")
    .to_frame()
)
categ_table["share"] = (categ_table["rows"] / categ_table["rows"].sum() * 100).round(1)
categ_table["meaning"] = [
    "No category (only examiner searches assign categories)" if code == "—" else explain_categ(code)
    for code in categ_table.index
]

categ_table.head(15).style.format({"share": "{:.1f}%"}).set_properties(subset=["meaning"], **{"text-align": "left"})

,rows,share,meaning
citn_categ,,,
A,20400,33.9%,Background
Y,11313,18.8%,Relevant combined with another document
X,11148,18.5%,Very relevant on its own
—,9943,16.5%,No category (only examiner searches assign categories)
I,5984,10.0%,⚠️ check DOCDB Annex XIV
AD,273,0.5%,Background + Already cited in the application
PX,236,0.4%,Published between priority and filing date + Very relevant on its own
E,205,0.3%,Earlier patent published on/after filing date
XP,140,0.2%,Very relevant on its own + Published between priority and filing date


### 6.3 Forward citations (`TLS212`, reversed)

Here we look **forward**: which **newer** patents refer to **our** drone patents.

In the citations table (TLS212) we look for rows that point **to** one of our patents.
Then we look up the newer patent that made the reference (TLS211).

A patent that many newer patents refer to is **influential**: other inventors build on it.
This is the **Influence** score of Step 3. To be fair, each patent is compared only with patents
of the **same technology and the same year**.

Careful: **recent** patents have had little time to be cited yet, so they will look less influential.

In [20]:
fwd = query_in_chunks(lambda ch: db.query(
    TLS212_CITATION.pat_publn_id.label("citing_pat_publn_id"),
    TLS212_CITATION.cited_pat_publn_id, TLS212_CITATION.citn_origin,
).filter(TLS212_CITATION.cited_pat_publn_id.in_(ch)), pubs["pat_publn_id"])

citing = query_in_chunks(lambda ch: db.query(
    TLS211_PAT_PUBLN.pat_publn_id.label("citing_pat_publn_id"),
    TLS211_PAT_PUBLN.appln_id.label("citing_appln_id"),
    TLS211_PAT_PUBLN.publn_auth.label("citing_auth"),
    TLS211_PAT_PUBLN.publn_date.label("citing_date"),
).filter(TLS211_PAT_PUBLN.pat_publn_id.in_(ch)), fwd["citing_pat_publn_id"])

fwd = fwd.merge(citing, on="citing_pat_publn_id")
fwd["citing_date"] = pd.to_datetime(fwd["citing_date"], errors="coerce")
if CITATIONS_EP_ONLY:
    fwd = fwd[fwd["citing_auth"].str.strip() == "EP"]
fwd["cited_appln_id"] = fwd["cited_pat_publn_id"].map(pubs.set_index("pat_publn_id")["appln_id"])
fwd = fwd[fwd["citing_appln_id"] != fwd["cited_appln_id"]]
fwd["citing_in_sector"] = fwd["citing_appln_id"].isin(appln_ids)
save(fwd, "tls212_forward")

received = fwd.drop_duplicates(["cited_appln_id", "citing_appln_id"]).groupby("cited_appln_id").size()
print(f"Sector EP applications cited at least once: {len(received):,} of {cit_appln['appln_id'].nunique():,}")

tls212_forward              2,071 rows  ->  /home/jovyan/progetto/extract/prod/tls212_forward.parquet
Sector EP applications cited at least once: 878 of 6,772


In [21]:
print("citations without a valid date:", fwd["citing_date"].isna().sum())
print("date range:", fwd["citing_date"].min(), "→", fwd["citing_date"].max())

citations without a valid date: 0
date range: 2006-05-17 00:00:00 → 2026-08-19 00:00:00


## 7. Legal Events

In [22]:
# Legal events: for Step 3 globalization refinement (tutorial_4/40 §7.2) and Step 6 (patent still in force?)

ep_granted = appln.loc[
    (appln["appln_auth"] == "EP") & (appln["granted"] == "Y"), "appln_id"
]

LEGAL_EVENT_CODES = ["PGFP", "VSFP", "PG25"]

events = query_in_chunks(lambda ch: db.query(
    TLS231_INPADOC_LEGAL_EVENT.appln_id,
    TLS231_INPADOC_LEGAL_EVENT.event_code,
    TLS231_INPADOC_LEGAL_EVENT.fee_country,
    TLS231_INPADOC_LEGAL_EVENT.lapse_country,
    TLS231_INPADOC_LEGAL_EVENT.lapse_text,
).filter(TLS231_INPADOC_LEGAL_EVENT.appln_id.in_(ch))
 .filter(TLS231_INPADOC_LEGAL_EVENT.event_code.in_(LEGAL_EVENT_CODES)), ep_granted)

events["lapse_text"] = events["lapse_text"].fillna("").str.upper()
events = events.merge(appln[["appln_id", "docdb_family_id"]], on="appln_id")
save(events, "tls231_legal_events")

print(f"Granted EP applications: {ep_granted.nunique():,}")
print("Events by code:", events["event_code"].value_counts().to_dict())
print("\nMost common lapse reasons (PG25):")
print(events.loc[events["event_code"] == "PG25", "lapse_text"].str[:70].value_counts().head(5).to_string())

tls231_legal_events       111,759 rows  ->  /home/jovyan/progetto/extract/prod/tls231_legal_events.parquet
Granted EP applications: 3,447
Events by code: {'PG25': 100385, 'PGFP': 11370, 'VSFP': 4}

Most common lapse reasons (PG25):
lapse_text
LAPSE BECAUSE OF FAILURE TO SUBMIT A TRANSLATION OF THE DESCRIPTION OR    83144
LAPSE BECAUSE OF NON-PAYMENT OF DUE FEES                                  17175
THE PATENT HAS BEEN ANNULLED BY A DECISION OF A NATIONAL AUTHORITY           39
LAPSE BECAUSE OF EXPIRATION OF PROTECTION                                    26
LAPSE BECAUSE OF THE APPLICANT RENOUNCES                                      1


## 8. Summary

In [23]:
pd.Series({
    "environment": ENV,
    "families": len(families),
    "applications (all members)": len(appln),
    "CPC rows": len(cpc),
    "person rows": len(persons),
    "families with English abstract": len(family_text),
    "EP publications": len(pubs),
    "backward citations": len(back),
    "forward citations (EP)": len(fwd),
}, name="extracted")

environment                          PROD
families                           115491
applications (all members)         162141
CPC rows                          1212309
person rows                        766421
families with English abstract     115487
EP publications                     13199
backward citations                  48432
forward citations (EP)               2071
Name: extracted, dtype: object